# Interrupts

Interrupts allow you to pause graph execution at specific points and wait for external input before continuing. This enables human-in-the-loop patterns where you need external input to proceed. When an interrupt is triggered, LangGraph saves the graph state using its `persistence` layer and waits indefinitely until you resume execution.

Interrupts work by calling the `interrupt`() function at any point in your graph nodes. The function accepts any JSON-serializable value which is surfaced to the caller. When you’re ready to continue, you resume execution by re-invoking the graph using `Command`, which then becomes the return value of the `interrupt`() call from inside the node.

Unlike static breakpoints (which pause before or after specific nodes), interrupts are **dynamic**: they can be placed anywhere in your code and can be conditional based on your application logic.

* Checkpointing keeps your place: the checkpointer writes the exact graph state so you can resume later, even when in an error state.
* thread_id is your pointer: set config={"configurable": {"thread_id": ...}} to tell the checkpointer which state to load.
* Interrupt payloads surface via chunk["interrupts"]: when streaming with version="v2", the values you pass to interrupt() appear in the interrupts field of values stream parts so you know what the graph is waiting on.
* The thread_id you choose is effectively your persistent cursor. Reusing it resumes the same checkpoint; using a new value starts a brand-new thread with an empty state.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## Pause using interrupt

The `interrupt` function pauses graph execution and returns a value to the caller. When you call `interrupt` within a node, LangGraph saves the current graph state and waits for you to resume execution with input.

To use `interrupt`, you need:

* A **checkpointer** to persist the graph state (use a durable checkpointer in production)
* A **thread ID** in your config so the runtime knows which state to resume from
* To call `interrupt`() where you want to pause (payload must be JSON-serializable)

```python
from langgraph.types import interrupt

def approval_node(state: State):
    # Pause and ask for approval
    approved = interrupt("Do you approve this action?")

    # When you resume, Command(resume=...) returns that value here
    return {"approved": approved}
```

When you call `interrupt`, here’s what happens:

1. **Graph execution gets suspended** at the exact point where `interrupt` is called
2. **State is saved** using the checkpointer so execution can be resumed later, In production, this should be a persistent checkpointer (e.g. backed by a database)
3. **Value is returned** to the caller under `__interrupt__`; it can be any JSON-serializable value (string, object, array, etc.)
4. **Graph waits indefinitely** until you resume execution with a response
5. **Response is passed back** into the node when you resume, becoming the return value of the `interrupt`() call

## Resuming interrupts

After an interrupt pauses execution, you resume the graph by invoking it again with a `Command` that contains the resume value. The resume value is passed back to the `interrupt` call, allowing the node to continue execution with the external input.

```python
from langgraph.types import Command

# Initial run - hits the interrupt and pauses
# thread_id is the persistent pointer (stores a stable ID in production)
config = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke({"input": "data"}, config=config, version="v2")

# result is a GraphOutput with .value and .interrupts
# .interrupts contains the payloads passed to interrupt()
print(result.interrupts)
# > (Interrupt(value='Do you approve this action?'),)

# Resume with the human's response
# The resume payload becomes the return value of interrupt() inside the node
graph.invoke(Command(resume=True), config=config, version="v2")
```

Key points about resuming:

* You must use the **same thread ID** when resuming that was used when the `interrupt` occurred
* The value passed to `Command(resume=...)` becomes the return value of the `interrupt` call
* The node restarts from the beginning of the node where the `interrupt` was called when resumed, so any code before the `interrupt` runs again
* You can pass any **JSON-serializable** value as the resume value

## Stream with human-in-the-loop (HITL) interrupts

When building interactive agents with human-in-the-loop workflows, you can stream both message chunks and node updates simultaneously to provide real-time feedback while handling interrupts.

Use multiple stream modes ("`messages`" and "`updates`") with `subgraphs=True` (if subgraphs are present) to:
* Stream AI responses in real-time as they’re generated
* Detect when the graph encounters an interrupt
* Handle user input and resume execution seamlessly

## Handling multiple interrupts

When parallel branches interrupt simultaneously (for example, fan-out to multiple nodes that each call `interrupt`()), you may need to resume multiple interrupts in a single invocation. When resuming multiple interrupts with a single invocation, map each interrupt ID to its resume value. This ensures each response is paired with the correct interrupt at runtime.

In [12]:
from typing import Annotated, TypedDict
import operator

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.types import Command, interrupt


class State(TypedDict):
    vals: Annotated[list[str], operator.add]


def node_a(state):
    answer = interrupt("question_a")
    return {"vals": [f"a:{answer}"]}


def node_b(state):
    answer = interrupt("question_b")
    return {"vals": [f"b:{answer}"]}


graph = (
    StateGraph(State)
    .add_node("a", node_a)
    .add_node("b", node_b)
    .add_edge(START, "a")
    .add_edge(START, "b")
    .add_edge("a", END)
    .add_edge("b", END)
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# Step 1: invoke - both parallel nodes hit interrupt() and pause
interrupted_result = graph.invoke({"vals": []}, config)
# print(interrupted_result)
"""
{
    'vals': [],
    '__interrupt__': [
        Interrupt(value='question_a', id='bd4f3183600f2c41dddafbf8f0f7be7b'),
        Interrupt(value='question_b', id='29963e3d3585f0cef025dd0f14323f55')
    ]
}
"""

"\n{\n    'vals': [],\n    '__interrupt__': [\n        Interrupt(value='question_a', id='bd4f3183600f2c41dddafbf8f0f7be7b'),\n        Interrupt(value='question_b', id='29963e3d3585f0cef025dd0f14323f55')\n    ]\n}\n"

In [13]:
# Step 2: resume all pending interrupts at once
resume_map = {
    i.id: f"answer for {i.value}" for i in interrupted_result["__interrupt__"]
}
result = graph.invoke(Command(resume=resume_map), config)

print("Final state:", result)
# > Final state: {'vals': ['a:answer for question_a', 'b:answer for question_b']}

Final state: {'vals': ['a:answer for question_a', 'b:answer for question_b']}


## Approve or reject

One of the most common uses of interrupts is to pause before a critical action and ask for approval. For example, you might want to ask a human to approve an API call, a database change, or any other important decision

In [ ]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt(
        {
            "question": "Approve this action?",
            "details": state["action_details"],
        }
    )

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}


builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

# Use a more durable checkpointer in production
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-123"}}
initial = graph.invoke(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
)
print(
    initial["__interrupt__"]
)  # -> [Interrupt(value={'question': ..., 'details': ...})]

[Interrupt(value={'question': 'Approve this action?', 'details': 'Transfer $500'}, id='1f99baae84db050c79ba744ad745a0b5')]


In [21]:
# Resume with the decision; True routes to proceed, False to cancel
resumed = graph.invoke(Command(resume=True), config=config)
# print(resumed["status"])  # -> "approved"
resumed

{'action_details': 'Transfer $500', 'status': 'approved'}

## Review and edit state

Sometimes you want to let a human review and edit part of the graph state before continuing. This is useful for correcting LLMs, adding missing information, or making adjustments.

In [26]:
import sqlite3
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class ReviewState(TypedDict):
    generated_text: str


def review_node(state: ReviewState):
    # Ask a reviewer to edit the generated content
    updated = interrupt(
        {
            "instruction": "Review and edit this content",
            "content": state["generated_text"],
        }
    )
    return {"generated_text": updated}


builder = StateGraph(ReviewState)
builder.add_node("review", review_node)
builder.add_edge(START, "review")
builder.add_edge("review", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "review-42"}}
initial = graph.invoke({"generated_text": "Initial draft"}, config=config)
# print(
#     initial["__interrupt__"]
# )  # -> [Interrupt(value={'instruction': ..., 'content': ...})]
print(initial)

# Resume with the edited text from the reviewer
final_state = graph.invoke(
    Command(resume="Improved draft after review"),
    config=config,
)
# print(final_state["generated_text"])  # -> "Improved draft after review"
final_state

{'generated_text': 'Initial draft', '__interrupt__': [Interrupt(value={'instruction': 'Review and edit this content', 'content': 'Initial draft'}, id='14dbb2c58fafa09074b1717d004d8267')]}


{'generated_text': 'Improved draft after review'}

## Interrupts in tools

You can also place `interrupts` directly inside tool functions. This makes the tool itself pause for approval whenever it’s called, and allows for human review and editing of the tool call before it is executed.\

First, define a tool that uses `interrupt`:

This approach is useful when you want the approval logic to live with the tool itself, making it reusable across different parts of your graph. The LLM can call the tool naturally, and the interrupt will pause execution whenever the tool is invoked, allowing you to approve, edit, or cancel the action.

In [57]:
import sqlite3
from typing import TypedDict

from langchain.tools import tool
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class AgentState(TypedDict):
    messages: list[dict]


@tool
def send_email(to: str, subject: str, body: str):
    """Send an email to a recipient."""

    # Pause before sending; payload surfaces in result["__interrupt__"]
    response = interrupt(
        {
            "action": "send_email",
            "to": to,
            "subject": subject,
            "body": body,
            "message": "Approve sending this email?",
        }
    )

    if response.get("action") == "approve":
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)

        # Actually send the email (your implementation here)
        print(f"[send_email] to={final_to} subject={final_subject} body={final_body}")
        return f"Email sent to {final_to}"

    return "Email cancelled by user"


model = ChatAnthropic(model="claude-sonnet-4-6").bind_tools([send_email])


def agent_node(state: AgentState):
    # LLM may decide to call the tool; interrupt pauses before sending
    result = model.invoke(state["messages"])
    return {"messages": state["messages"] + [result]}


builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

conn = sqlite3.connect("tool-approval.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "email-workflow"}}
initial = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Send an email to alice@example.com with subject 'Meeting Tomorrow' and body 'Let's meet tomorrow at 10 AM.'",
            }
        ]
    },
    config=config,
)
# print(initial["__interrupt__"])
print(
    initial["messages"][-1].content
)  # -> [Interrupt(value={'action': 'send_email', ...})]

[{'id': 'toolu_01HzZyHRY6cfbK3mNqczVqFo', 'caller': {'type': 'direct'}, 'input': {'to': 'alice@example.com', 'subject': 'Meeting Tomorrow', 'body': "Let's meet tomorrow at 10 AM."}, 'name': 'send_email', 'type': 'tool_use'}]


In [59]:
# Resume with approval and optionally edited arguments
resumed = graph.invoke(
    Command(
        resume={
            "action": "reject",
            # "subject": "Updated subject",
            # "body": "Updated body",
        }
    ),
    config=config,
)
print(resumed["messages"][-1].content)  # -> Tool result returned by send_email

[{'id': 'toolu_01HzZyHRY6cfbK3mNqczVqFo', 'caller': {'type': 'direct'}, 'input': {'to': 'alice@example.com', 'subject': 'Meeting Tomorrow', 'body': "Let's meet tomorrow at 10 AM."}, 'name': 'send_email', 'type': 'tool_use'}]


In [60]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("tool-approval.db")

df = pd.read_sql("SELECT * FROM checkpoints", conn)
print(df)

         thread_id checkpoint_ns                         checkpoint_id  \
0   email-workflow                1f121ded-c4dc-6796-bfff-71b824ba0806   
1   email-workflow                1f121ded-c4e0-6a44-8000-51cd095bc41b   
2   email-workflow                1f121ded-d5fa-6262-8001-cfe1253e85b6   
3   email-workflow                1f121dee-2513-621d-8002-3a9173358299   
4   email-workflow                1f121dee-2515-6d3a-8003-c138250d6c49   
5   email-workflow                1f121dee-3253-6653-8004-f9b74d0ad9a7   
6   email-workflow                1f121df2-6a7c-690c-8005-ae5fe0eab66f   
7   email-workflow                1f121df2-6a86-6602-8006-3f184a78e262   
8   email-workflow                1f121df2-7795-66f3-8007-cbc876676d75   
9   email-workflow                1f121df3-04c3-65cc-8008-5d40884d443a   
10  email-workflow                1f121df3-04c8-6446-8009-3ce844d29416   
11  email-workflow                1f121df3-10ba-650f-800a-d465fe5ce0b9   
12  email-workflow                1f12

## Validating human input

Sometimes you need to validate input from humans and ask again if it’s invalid. You can do this using multiple `interrupt` calls in a loop.

Each time you resume the graph with invalid input, it will ask again with a clearer message. Once valid input is provided, the node completes and the graph continues.

In [63]:
import sqlite3
from typing import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class FormState(TypedDict):
    age: int | None


def get_age_node(state: FormState):
    prompt = "What is your age?"

    while True:
        answer = interrupt(prompt)  # payload surfaces in result["__interrupt__"]

        if isinstance(answer, int) and answer > 0:
            return {"age": answer}

        prompt = f"'{answer}' is not a valid age. Please enter a positive number."


builder = StateGraph(FormState)
builder.add_node("collect_age", get_age_node)
builder.add_edge(START, "collect_age")
builder.add_edge("collect_age", END)

# checkpointer = SqliteSaver(sqlite3.connect("forms.db"))
conn = sqlite3.connect("forms.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "form-1"}}
first = graph.invoke({"age": None}, config=config)
print(first["__interrupt__"])  # -> [Interrupt(value='What is your age?', ...)]

# Provide invalid data; the node re-prompts
retry = graph.invoke(Command(resume="thirty"), config=config)
print(
    retry["__interrupt__"]
)  # -> [Interrupt(value="'thirty' is not a valid age...", ...)]

# Provide valid data; loop exits and state updates
final = graph.invoke(Command(resume=30), config=config)
print(final["age"])  # -> 30

[Interrupt(value='What is your age?', id='6972cb990db1919d7b68a4329284d0c1')]
[Interrupt(value="'thirty' is not a valid age. Please enter a positive number.", id='6972cb990db1919d7b68a4329284d0c1')]
30
